# Gold Table: gold.top_videos

## Objective

Create a ranked dataset containing the highest-performing YouTube videos.

## Source
- gold.video_performance

## Grain
- One row per ranked video

## Ranking

- Rank videos by Views (Descending)
- Retain the Top 10 videos

## Columns

| Column | Description |
|---------|-------------|
| rank | Video ranking based on views |
| video_id | Unique video identifier |
| title | Video title |
| video_type | Short or Normal |
| duration | Video duration |
| views | Total views |
| likes | Total likes |
| comments | Total comments |
| engagement_rate | ((Likes + Comments) / Views) × 100 |

## Business Purpose

- Identify the best-performing videos.
- Highlight top content for reporting.
- Support Top 10 dashboard visualizations.

In [0]:
from pyspark.sql import functions as f
from pyspark.sql import Window

In [0]:
source_table="youtube.gold.video_performance"
target_table="youtube.gold.top_videos"

In [0]:
top_videos_df = (
    spark.read.table(source_table)
    .withColumn("views", f.col("views").cast("long"))
    .withColumn("likes", f.col("likes").cast("long"))
    .withColumn("comments", f.col("comments").cast("long"))
    .withColumn("rank", f.row_number().over(Window.orderBy(f.desc("views"))))
    .filter(f.col("rank") <= 10)
    .select(
        "rank",
        "video_id",
        "title",
        "video_type",
        "duration",
        "views",
        "likes",
        "comments",
        f.round(
            ((f.col("likes") + f.col("comments")) / f.col("views")) * 100,
            2
        ).alias("engagement_rate")
    )
)
display(top_videos_df)